In [ ]:
from sympy import symbols, sin, pi, lambdify
import numpy as np
import matplotlib.pyplot as plt


## PLOT 1, 2D
theta, r = symbols('theta r')
boundary_expr = sin(4 * theta) * sin(2 * theta)
boundary_fn = lambdify(theta, boundary_expr, modules=['numpy'])
theta_vals = np.linspace(0, 2*np.pi, 500)
boundary_vals = boundary_fn(theta_vals)
plt.figure(figsize=(6, 3))
plt.plot(theta_vals, boundary_vals)
plt.title("Spiral Boundary Waveform")
plt.xlabel("theta")
plt.ylabel("Amplitude")
plt.grid(True)
plt.show()

## PLOT 2, 3D
r_vals = np.linspace(0, 1, 300)
r_grid, theta_grid = np.meshgrid(r_vals, theta_vals)
boundary_wave = boundary_fn(theta_grid)
interior = boundary_wave * np.cos(np.pi * r_grid)**2
x = r_grid * np.cos(theta_grid)
y = r_grid * np.sin(theta_grid)
plt.figure(figsize=(6, 6))
plt.contourf(x, y, interior, 100, cmap="plasma")
plt.title("Interior Field Reconstruction from Spiral Boundary")
plt.axis('equal')
plt.colorbar(label="Signal Density")
plt.show()

In [ ]:
from sympy import symbols, sin, cos, pi, lambdify
import numpy as np
import matplotlib.pyplot as plt
theta, r = symbols('theta r')
# boundary_expr = sin(4 * theta) * sin(2 * theta)
boundary_expr = cos(theta)**2
boundary_fn = lambdify(theta, boundary_expr, modules=['numpy'])
theta_vals = np.linspace(0, 2*np.pi, 500)
boundary_vals = boundary_fn(theta_vals)
plt.figure(figsize=(6, 3))
plt.plot(theta_vals, boundary_vals)
plt.title("Spiral Boundary Waveform")
plt.xlabel("theta")
plt.ylabel("Amplitude")
plt.grid(True)
plt.show()

In [ ]:
r_vals = np.linspace(0, 1, 300)
r_grid, theta_grid = np.meshgrid(r_vals, theta_vals)
boundary_wave = boundary_fn(theta_grid)
interior = boundary_wave * np.cos(np.pi * r_grid)
x = r_grid * np.cos(theta_grid)
y = r_grid * np.sin(theta_grid)
plt.figure(figsize=(6, 6))
plt.contourf(x, y, interior, 100, cmap="plasma")
plt.title("Interior Field Reconstruction from Spiral Boundary")
plt.axis('equal')
plt.colorbar(label="Signal Density")
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# ----------------------------
# Photon entangled-pair torus texture and its "collapsed" disk view
# For linear polarizers: P(opposite) ~ cos^2(Δ) with Δ = θA - θB
# We'll visualize W(θA,θB) ∝ cos^2(θA - θB).
# Then show that your disk is essentially W(Δ) with Δ mapped to radius r.
# ----------------------------

# Torus grid
N = 520
thetaA = np.linspace(0, 2*np.pi, N, endpoint=False)
thetaB = np.linspace(0, 2*np.pi, N, endpoint=False)
TA, TB = np.meshgrid(thetaA, thetaB, indexing="xy")

Delta = (TA - TB)  # relative angle on the torus
W = np.cos(Delta)**2  # photon "opposite" probability-like texture (up to labeling conventions)

# Normalize for display
Wn = (W - W.min()) / (W.max() - W.min() + 1e-12)

# 1D profile vs Δ (collapse sum-coordinate by averaging over u ≡ (θA+θB)/2)
# Since W depends only on Δ, this average just reproduces cos^2(Δ).
delta_line = np.linspace(0, 2*np.pi, 1000, endpoint=False)
W_delta = np.cos(delta_line)**2

# Disk view: map radius r ∈ [0,1] to Δ ∈ [0,π] (unique range for cos^2 periodicity)
Nr, Nth = 340, 700
r_vals = np.linspace(0, 1, Nr)
th_vals = np.linspace(0, 2*np.pi, Nth, endpoint=False)
r_grid, th_grid = np.meshgrid(r_vals, th_vals, indexing="xy")
Delta_disk = np.pi * r_grid  # Δ = π r
W_disk = np.cos(Delta_disk)**2
W_disk_n = (W_disk - W_disk.min()) / (W_disk.max() - W_disk.min() + 1e-12)

x = r_grid * np.cos(th_grid)
y = r_grid * np.sin(th_grid)

# ----------------------------
# Plot: torus heatmap, disk collapsed view, and Δ profile
# ----------------------------
fig = plt.figure(figsize=(14, 4.8), constrained_layout=True)
gs = fig.add_gridspec(1, 3, width_ratios=[1.2, 1.0, 1.0])

ticks = [0, np.pi/2, np.pi, 3*np.pi/2, 2*np.pi]
ticklabels = ["0", r"$\pi/2$", r"$\pi$", r"$3\pi/2$", r"$2\pi$"]

# (1) Torus (unwrapped) heatmap
ax0 = fig.add_subplot(gs[0, 0])
im = ax0.imshow(
    Wn,
    origin="lower",
    extent=(0, 2*np.pi, 0, 2*np.pi),
    interpolation="nearest",
    aspect="equal",
)
ax0.set_title(r"Torus texture (photons): $W(\theta_A,\theta_B)=\cos^2(\theta_A-\theta_B)$")
ax0.set_xlabel(r"$\theta_A$")
ax0.set_ylabel(r"$\theta_B$")
ax0.set_xticks(ticks, ticklabels)
ax0.set_yticks(ticks, ticklabels)
fig.colorbar(im, ax=ax0, fraction=0.046, pad=0.04, label="normalized")

# (2) Disk collapsed view (Δ collapsed into radius)
ax1 = fig.add_subplot(gs[0, 1])
cf = ax1.contourf(x, y, W_disk_n, 120)
ax1.set_aspect("equal", adjustable="box")
ax1.set_title(r"Disk view: radius encodes $\Delta$ via $\Delta=\pi r$")
ax1.set_xlabel("x")
ax1.set_ylabel("y")
ax1.set_xticks([])
ax1.set_yticks([])
fig.colorbar(cf, ax=ax1, fraction=0.046, pad=0.04, label="normalized")

# draw a few Δ reference rings
deg = np.pi / 180
for ddeg in [0, 22.5, 45, 67.5, 90, 135, 180]:
    rr = (ddeg * deg) / np.pi
    if 0 < rr <= 1:
        ax1.add_patch(plt.Circle((0, 0), rr, fill=False, linewidth=1.0, alpha=0.55))

# (3) 1D profile W(Δ)
ax2 = fig.add_subplot(gs[0, 2])
ax2.plot(delta_line, W_delta, lw=2)
ax2.set_title(r"Collapsed profile: $W(\Delta)=\cos^2(\Delta)$")
ax2.set_xlabel(r"$\Delta=\theta_A-\theta_B$")
ax2.set_ylabel("W")
ax2.set_xticks(ticks, ticklabels)
ax2.grid(True, alpha=0.25)

# Mark Δ=22.5° and 45° (common CHSH angles for photons and spin comparisons)
for ddeg in [22.5, 45, 90]:
    dd = ddeg * deg
    ax2.axvline(dd, linewidth=1.0, alpha=0.6)
    ax2.text(dd, 1.02, f"{ddeg:g}°", ha="center", va="bottom", fontsize=9)

fig.suptitle(
    "Same information in three coordinate systems: torus (θA,θB) → relative angle Δ → disk (Δ collapsed to radius)",
    fontsize=13
)
plt.show()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# ----------------------------------------
# "Hidden geometry" disk model:
# - Amplitude (Born weight) depends only on relative angle Δ, mapped to radius r via Δ = π r
#   For entangled photons (linear polarizers): choose R^2(r) = cos^2(Δ) = cos^2(π r)
# - Phase encodes a "winding" in the unused/common-mode direction (disk angle θ), like a gauge/helix texture:
#   S(r,θ) = m θ + k r  (simple helical phase sheet)
# - Complex field: Ψ = R * exp(i S)
# This keeps probabilities tied to Δ while giving you a nontrivial hidden phase geometry.
# ----------------------------------------

Nr, Nth = 360, 900
r_vals = np.linspace(0, 4, Nr)
th_vals = np.linspace(0, 6*np.pi, Nth, endpoint=False)
r, th = np.meshgrid(r_vals, th_vals, indexing="xy")

# Map radius -> relative analyzer angle
Delta = np.pi * r  # Δ in [0, π]

# Amplitude / Born weight (choose photon-style envelope)
R2 = np.cos(Delta)**2
R = np.sqrt(R2)

# Helical phase texture (hidden geometry)
m = 2        # azimuthal winding number (how many wraps around the disk)
k = np.pi  # radial winding (adds a spiral twist)
S = (m * th + k * r) % (2*np.pi)

Psi = R * np.exp(1j * S)

# Convert to Cartesian for plotting
x = r * np.cos(th)
y = r * np.sin(th)

# Helper: normalize to 0..1
def norm01(z):
    zmin, zmax = z.min(), z.max()
    return (z - zmin) / (zmax - zmin + 1e-12)

# Panels to show:
# 1) Born weight R^2 (what affects counts)
# 2) Phase S (hidden geometry; cyclic)
# 3) Re(Ψ) (shows interference-like structure combining amplitude+phase)
R2n = norm01(R2)
RePsi = np.real(Psi)
RePsi_n = norm01(RePsi)

fig = plt.figure(figsize=(13.2, 4.8), constrained_layout=True)
gs = fig.add_gridspec(1, 3, width_ratios=[1, 1, 1])

# Panel 1: Born weight
ax0 = fig.add_subplot(gs[0, 0])
cf0 = ax0.contourf(x, y, R2n, 140)
ax0.set_aspect("equal", adjustable="box")
ax0.set_title(r"Born weight (counts): $R^2=\cos^2(\Delta)$ with $\Delta=\pi r$")
ax0.set_xticks([]); ax0.set_yticks([])
fig.colorbar(cf0, ax=ax0, fraction=0.046, pad=0.04, label="normalized")

# Add Δ reference rings
# deg = np.pi / 180
# for ddeg in [0, 22.5, 45, 67.5, 90, 135, 180]:
#     rr = (ddeg * deg) / np.pi
#     if 0 < rr <= 1:
#         ax0.add_patch(plt.Circle((0, 0), rr, fill=False, linewidth=1.0, alpha=0.55))

# Panel 2: Phase (cyclic colormap)
ax1 = fig.add_subplot(gs[0, 1])
# Use hsv for cyclic phase; do not specify custom colors beyond default map name
cf1 = ax1.contourf(x, y, S, 160, cmap="hsv")
ax1.set_aspect("equal", adjustable="box")
ax1.set_title(r"Hidden geometry: phase field $S(r,\theta)=m\theta+kr$ (wrapped)")
ax1.set_xticks([]); ax1.set_yticks([])
cb1 = fig.colorbar(cf1, ax=ax1, fraction=0.046, pad=0.04)
cb1.set_label("phase (radians)")

# Panel 3: Real part of Ψ
ax2 = fig.add_subplot(gs[0, 2])
cf2 = ax2.contourf(x, y, RePsi_n, 160)
ax2.set_aspect("equal", adjustable="box")
ax2.set_title(r"One observable slice: $\Re[\Psi]=R\cos S$ (amplitude × phase)")
ax2.set_xticks([]); ax2.set_yticks([])
fig.colorbar(cf2, ax=ax2, fraction=0.046, pad=0.04, label="normalized")

fig.suptitle(
    "Hidden-geometry disk: keep probabilities radial in Δ, put nontrivial structure in the phase",
    fontsize=13
)
plt.show()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize

# ------------------------------------------------------------
# Global joint phase on the torus (configuration-space pilot-wave vibe)
#
# Coordinates:
#   θA, θB ∈ [0,2π)
#   u = (θA + θB)/2   (common-mode / global rotation coordinate)
#   v = (θA - θB)/2   (relative coordinate)
#
# Photon-polarizer-style Born weight depends on Δ = θA - θB = 2v:
#   R^2(θA,θB) = cos^2(Δ) = cos^2(2v)
#
# Pilot-wave "hidden geometry" lives in the PHASE S(u,v).
# We'll choose a phase with:
#   - a global winding in u (m u): a joint phase helix around the torus
#   - a coupling term in v (alpha sin(2v)): makes the phase gradient depend on the relative coordinate
#
# Complex field:
#   Ψ(θA,θB) = R(v) * exp(i S(u,v))
#
# Then show a disk projection:
#   disk angle θ_disk := u
#   disk radius r := v / (π/2)   (since v ∈ [0,π) modulo, use [0,π/2] as principal for Δ∈[0,π])
# ------------------------------------------------------------

# Grid on torus
N = 520
thetaA = np.linspace(0, 2*np.pi, N, endpoint=False)
thetaB = np.linspace(0, 2*np.pi, N, endpoint=False)
TA, TB = np.meshgrid(thetaA, thetaB, indexing="xy")

u = 0.5 * (TA + TB)
v = 0.5 * (TA - TB)  # relative half-angle

# Wrap v into [-pi, pi) for nicer phase features (periodic anyway)
v_wrapped = (v + np.pi) % (2*np.pi) - np.pi

# Born weight for photons (Δ = θA-θB = 2v)
R2 = np.cos(2 * v_wrapped)**2
R = np.sqrt(R2)

# Joint phase field S(u,v): global + coupling
m = 2            # global winding count around u
alpha = 1.2      # relative-coupling strength
beta = 0.4       # optional small u-v coupling
S = (m * u + alpha * np.sin(2 * v_wrapped) + beta * np.sin(u) * np.cos(2*v_wrapped)) % (2*np.pi)

Psi = R * np.exp(1j * S)

# Display helpers
def norm01(z):
    zmin, zmax = z.min(), z.max()
    return (z - zmin) / (zmax - zmin + 1e-12)

R2n = norm01(R2)
RePsi = np.real(Psi)
RePsi_n = norm01(RePsi)

# ----------------------------------------
# Disk projection of (u,v) onto (r,theta)
# Choose principal range Δ ∈ [0,π] -> v ∈ [0,π/2]
# Map: r = |v| / (π/2), theta = u
# This "collapses" the torus into a disk while preserving:
#   - angular coordinate = common-mode u
#   - radial coordinate = relative coordinate magnitude |v|
# ----------------------------------------
# Build disk grid in (r,theta_disk)
Nr, Nth = 360, 900
r_vals = np.linspace(0, 1, Nr)
th_vals = np.linspace(0, 2*np.pi, Nth, endpoint=False)
r_grid, th_grid = np.meshgrid(r_vals, th_vals, indexing="xy")

u_disk = th_grid
v_mag = r_grid * (np.pi/2)  # |v| in [0, π/2]
# Use the same functional forms but with v = ±v_mag; since cos^2(2v) and sin(2v) are even/odd,
# we can pick v = v_mag for a consistent disk "branch".
v_disk = v_mag

R2_disk = np.cos(2 * v_disk)**2
R_disk = np.sqrt(R2_disk)
S_disk = (m * u_disk + alpha * np.sin(2 * v_disk) + beta * np.sin(u_disk) * np.cos(2*v_disk)) % (2*np.pi)
Psi_disk = R_disk * np.exp(1j * S_disk)

x = r_grid * np.cos(th_grid)
y = r_grid * np.sin(th_grid)

R2_disk_n = norm01(R2_disk)
S_disk_n = S_disk  # already [0,2π)
RePsi_disk_n = norm01(np.real(Psi_disk))

# ----------------------------------------
# Plot
# ----------------------------------------
fig = plt.figure(figsize=(14, 8), constrained_layout=True)
gs = fig.add_gridspec(2, 3, width_ratios=[1.15, 1.15, 1.0], height_ratios=[1, 1])

ticks = [0, np.pi/2, np.pi, 3*np.pi/2, 2*np.pi]
ticklabels = ["0", r"$\pi/2$", r"$\pi$", r"$3\pi/2$", r"$2\pi$"]

# Torus: Born weight
ax0 = fig.add_subplot(gs[0, 0])
im0 = ax0.imshow(R2n, origin="lower", extent=(0, 2*np.pi, 0, 2*np.pi), aspect="equal", interpolation="nearest")
ax0.set_title(r"Torus: Born weight $R^2(\theta_A,\theta_B)=\cos^2(\theta_A-\theta_B)$")
ax0.set_xlabel(r"$\theta_A$")
ax0.set_ylabel(r"$\theta_B$")
ax0.set_xticks(ticks, ticklabels)
ax0.set_yticks(ticks, ticklabels)
fig.colorbar(im0, ax=ax0, fraction=0.046, pad=0.04, label="normalized")

# Torus: phase field (hidden geometry)
ax1 = fig.add_subplot(gs[0, 1])
im1 = ax1.imshow(S, origin="lower", extent=(0, 2*np.pi, 0, 2*np.pi), aspect="equal", interpolation="nearest", cmap="hsv")
ax1.set_title(r"Torus: joint phase $S(u,v)$ (global pilot-wave geometry)")
ax1.set_xlabel(r"$\theta_A$")
ax1.set_ylabel(r"$\theta_B$")
ax1.set_xticks(ticks, ticklabels)
ax1.set_yticks(ticks, ticklabels)
cb1 = fig.colorbar(im1, ax=ax1, fraction=0.046, pad=0.04)
cb1.set_label("phase (radians)")

# Torus: one observable slice Re[Ψ]
ax2 = fig.add_subplot(gs[0, 2])
im2 = ax2.imshow(RePsi_n, origin="lower", extent=(0, 2*np.pi, 0, 2*np.pi), aspect="equal", interpolation="nearest")
ax2.set_title(r"Torus: $\Re[\Psi]=R\cos S$")
ax2.set_xlabel(r"$\theta_A$")
ax2.set_ylabel(r"$\theta_B$")
ax2.set_xticks(ticks, ticklabels)
ax2.set_yticks(ticks, ticklabels)
fig.colorbar(im2, ax=ax2, fraction=0.046, pad=0.04, label="normalized")

# Disk: Born weight (radial in v)
ax3 = fig.add_subplot(gs[1, 0])
cf3 = ax3.contourf(x, y, R2_disk_n, 160)
ax3.set_aspect("equal", adjustable="box")
ax3.set_title(r"Disk projection: $R^2$ with $r\propto |v|$ and $\theta=u$")
ax3.set_xticks([]); ax3.set_yticks([])
fig.colorbar(cf3, ax=ax3, fraction=0.046, pad=0.04, label="normalized")

# Disk: phase field
ax4 = fig.add_subplot(gs[1, 1])
cf4 = ax4.contourf(x, y, S_disk_n, 180, cmap="hsv")
ax4.set_aspect("equal", adjustable="box")
ax4.set_title(r"Disk projection: phase $S(u,v)$")
ax4.set_xticks([]); ax4.set_yticks([])
cb4 = fig.colorbar(cf4, ax=ax4, fraction=0.046, pad=0.04)
cb4.set_label("phase (radians)")

# Disk: Re[Ψ]
ax5 = fig.add_subplot(gs[1, 2])
cf5 = ax5.contourf(x, y, RePsi_disk_n, 160)
ax5.set_aspect("equal", adjustable="box")
ax5.set_title(r"Disk projection: $\Re[\Psi]=R\cos S$")
ax5.set_xticks([]); ax5.set_yticks([])
fig.colorbar(cf5, ax=ax5, fraction=0.046, pad=0.04, label="normalized")

fig.suptitle(
    "Configuration-space pilot-wave vibe: amplitude set by relative angle, hidden joint geometry in a global phase field on the torus",
    fontsize=13
)
plt.show()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# ------------------------------------------------------------
# "Spiral boundary as photon" toy model (illustrative):
# Treat the boundary field as a *spatial mode* with orbital angular momentum (OAM)-like phase:
#   boundary(θ) = exp(i ℓ θ)
# Then treat polarization as a Jones vector (Ex, Ey):
#   H: (Ex, Ey) = (boundary, 0)
#   V: (Ex, Ey) = (0, boundary)
#
# Interior reconstruction (as in your original code):
#   interior(r,θ) = boundary(θ) * cos^2(pi r)
# applied component-wise to Ex, Ey.
#
# Visualizations:
# - Intensity S0 = |Ex|^2 + |Ey|^2  (what "counts" see)
# - Polarization angle ψ from Stokes (color hue), intensity as brightness
# - Real part Re(Ex) or Re(Ey) to show the spiral phase in a scalar plot
# ------------------------------------------------------------

Nr, Nth = 320, 720
r_vals = np.linspace(0, 1, Nr)
theta_vals = np.linspace(0, 2*np.pi, Nth, endpoint=False)
r, th = np.meshgrid(r_vals, theta_vals, indexing="xy")

# Spiral boundary mode (complex)
ell = 4
boundary = np.exp(1j * ell * th)

# Radial envelope (your cos^2)
env = np.cos(np.pi * r) ** 2

# Component-wise interior "reconstruction"
def reconstruct(jones):
    Ex, Ey = jones
    Ex_i = Ex * env
    Ey_i = Ey * env
    return Ex_i, Ey_i

# Jones vectors for H and V
ExH, EyH = boundary, 0*boundary
ExV, EyV = 0*boundary, boundary

ExH_i, EyH_i = reconstruct((ExH, EyH))
ExV_i, EyV_i = reconstruct((ExV, EyV))

# Also show a 45° linear polarization as a comparison (equal components)
Ex45, Ey45 = boundary/np.sqrt(2), boundary/np.sqrt(2)
Ex45_i, Ey45_i = reconstruct((Ex45, Ey45))

# Stokes parameters
def stokes(Ex, Ey):
    S0 = np.abs(Ex)**2 + np.abs(Ey)**2
    S1 = np.abs(Ex)**2 - np.abs(Ey)**2
    S2 = 2*np.real(Ex*np.conj(Ey))
    S3 = -2*np.imag(Ex*np.conj(Ey))
    return S0, S1, S2, S3

def pol_angle_from_stokes(S1, S2):
    # ψ in [-pi/2, pi/2), where ψ=0 is horizontal, ψ=pi/2 is vertical
    return 0.5*np.arctan2(S2, S1)

# Convert to disk coordinates
x = r * np.cos(th)
y = r * np.sin(th)

# Build "polarization-colored" RGB image: hue = ψ (wrapped), value = normalized S0
def hsv_to_rgb(h, s, v):
    # h,s,v arrays in [0,1]
    i = (h*6.0).astype(int) % 6
    f = (h*6.0) - np.floor(h*6.0)
    p = v*(1.0-s)
    q = v*(1.0-f*s)
    t = v*(1.0-(1.0-f)*s)
    r_ = np.choose(i, [v,q,p,p,t,v])
    g_ = np.choose(i, [t,v,v,q,p,p])
    b_ = np.choose(i, [p,p,t,v,v,q])
    return np.stack([r_, g_, b_], axis=-1)

def pol_rgb(Ex, Ey):
    S0, S1, S2, S3 = stokes(Ex, Ey)
    psi = pol_angle_from_stokes(S1, S2)   # radians
    # Map ψ (range ~[-pi/2, pi/2]) to hue [0,1]
    hue = ((psi + np.pi/2) / np.pi) % 1.0
    # Saturation: 1 for fully polarized (these examples are fully polarized)
    sat = np.ones_like(hue)
    # Value: normalized intensity
    val = S0 / (S0.max() + 1e-12)
    return hsv_to_rgb(hue, sat, val), S0, psi

rgbH, S0H, psiH = pol_rgb(ExH_i, EyH_i)
rgbV, S0V, psiV = pol_rgb(ExV_i, EyV_i)
rgb45, S045, psi45 = pol_rgb(Ex45_i, Ey45_i)

# Scalar "phase-visible" fields
ReH = np.real(ExH_i)
ReV = np.real(EyV_i)
Re45 = np.real(Ex45_i + Ey45_i)  # arbitrary scalar mix to show structure

# Normalize scalar fields for display
def norm01(z):
    return (z - z.min()) / (z.max() - z.min() + 1e-12)

ReHn, ReVn, Re45n = norm01(ReH), norm01(ReV), norm01(Re45)

# Plot
fig = plt.figure(figsize=(13.5, 7.2), constrained_layout=True)
gs = fig.add_gridspec(3, 2, height_ratios=[1, 1, 1])

# Row 1: H
ax = fig.add_subplot(gs[0, 0])
ax.imshow(rgbH, origin="lower", extent=(-1,1,-1,1))
ax.set_title("H photon (Jones: Ex≠0, Ey=0)\nHue = polarization angle ψ, Brightness = intensity |E|²")
ax.set_axis_off()

ax = fig.add_subplot(gs[0, 1])
ax.contourf(x, y, ReHn, 140)
ax.set_aspect("equal", adjustable="box")
ax.set_title("H photon: scalar view (Re[Ex]) shows spiral phase")
ax.set_axis_off()

# Row 2: V
ax = fig.add_subplot(gs[1, 0])
ax.imshow(rgbV, origin="lower", extent=(-1,1,-1,1))
ax.set_title("V photon (Jones: Ex=0, Ey≠0)\nHue differs (ψ≈90°) but intensity rings match")
ax.set_axis_off()

ax = fig.add_subplot(gs[1, 1])
ax.contourf(x, y, ReVn, 140)
ax.set_aspect("equal", adjustable="box")
ax.set_title("V photon: scalar view (Re[Ey]) shows same spiral phase")
ax.set_axis_off()

# Row 3: 45° comparison
ax = fig.add_subplot(gs[2, 0])
ax.imshow(rgb45, origin="lower", extent=(-1,1,-1,1))
ax.set_title("45° linear (Ex=Ey)\nHue sits midway between H and V")
ax.set_axis_off()

ax = fig.add_subplot(gs[2, 1])
ax.contourf(x, y, Re45n, 140)
ax.set_aspect("equal", adjustable="box")
ax.set_title("45°: scalar mix shows spiral interference-like texture")
ax.set_axis_off()

fig.suptitle(
    "Framing: spiral boundary = spatial mode; polarization = separate internal DOF (Jones vector).\n"
    "Counts (|E|²) don't care whether it's H or V, but a polarization-colored plot does.",
    fontsize=12
)
plt.show()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# ------------------------------------------------------------
# Synthetic SPDC visuals:
# 1) Two far-field "disks" (one per photon) showing a ring distribution.
# 2) Joint angular correlation P(theta_A, theta_B) showing opposite-point correlation.
# 3) Joint spectrum P(ω_A, ω_B) showing energy conservation ω_A + ω_B ≈ ω_p.
#
# This is a pedagogical, not experiment-calibrated, synthetic model.
# ------------------------------------------------------------

rng = np.random.default_rng(4)

# --- Parameters for far-field ring (transverse momentum / emission angle) ---
N_pairs = 12000
r0 = 0.78            # ring radius (normalized)
sigma_r = 0.1      # ring thickness
sigma_theta = 0.5   # angular correlation noise (radians); 0 => perfect opposite points

thetaA = rng.uniform(0, 2*np.pi, size=N_pairs)
thetaB = (thetaA + np.pi + rng.normal(0, sigma_theta, size=N_pairs)) % (2*np.pi)

rA = np.clip(r0 + rng.normal(0, sigma_r, size=N_pairs), 0, 1)
rB = np.clip(r0 + rng.normal(0, sigma_r, size=N_pairs), 0, 1)

xA, yA = rA * np.cos(thetaA), rA * np.sin(thetaA)
xB, yB = rB * np.cos(thetaB), rB * np.sin(thetaB)

# --- Parameters for joint spectrum (energy conservation) ---
# Use normalized units: ω_p = 2.0, with a narrow "sum" deviation and wider "difference" spread.
omega_p = 2.0
sigma_sum = 0.015    # thickness around ωA+ωB=ωp (pump bandwidth / phase matching)
sigma_diff = 0.14    # spread along the anti-diagonal direction

u = rng.normal(0, sigma_sum, size=N_pairs)   # deviation of (ωA+ωB)/2 from ωp/2
v = rng.normal(0, sigma_diff, size=N_pairs)  # (ωA-ωB)/2 spread

omegaA = (omega_p/2) + u + v
omegaB = (omega_p/2) + u - v

# Keep in a nice plotting window
wmin, wmax = 0.6, 1.4

# --- 2D histograms ---
# Joint angular correlation histogram
nb_ang = 160
H_ang, xedges_ang, yedges_ang = np.histogram2d(thetaA, thetaB, bins=nb_ang, range=[[0, 2*np.pi], [0, 2*np.pi]])
# Joint spectrum histogram
nb_w = 200
H_w, xedges_w, yedges_w = np.histogram2d(omegaA, omegaB, bins=nb_w, range=[[wmin, wmax], [wmin, wmax]])

# --- Plot layout ---
fig = plt.figure(figsize=(12.5, 9.2), constrained_layout=True)
gs = fig.add_gridspec(2, 2, width_ratios=[1, 1.05], height_ratios=[1, 1])

# Panel (0,0): Disk A
ax0 = fig.add_subplot(gs[0, 0])
ax0.scatter(xA, yA, s=1, alpha=0.15)
ax0.set_aspect("equal", adjustable="box")
ax0.set_xlim(-1.02, 1.02)
ax0.set_ylim(-1.02, 1.02)
ax0.set_xticks([]); ax0.set_yticks([])
ax0.set_title("Photon A far-field distribution (ring)")
# Draw unit circle for reference
circle = plt.Circle((0, 0), 1.0, fill=False, linewidth=1.0, alpha=0.5)
ax0.add_patch(circle)

# Panel (0,1): Disk B
ax1 = fig.add_subplot(gs[0, 1])
ax1.scatter(xB, yB, s=1, alpha=0.15)
ax1.set_aspect("equal", adjustable="box")
ax1.set_xlim(-1.02, 1.02)
ax1.set_ylim(-1.02, 1.02)
ax1.set_xticks([]); ax1.set_yticks([])
ax1.set_title("Photon B far-field distribution (ring)")
circle = plt.Circle((0, 0), 1.0, fill=False, linewidth=1.0, alpha=0.5)
ax1.add_patch(circle)

# Panel (1,0): Joint angular correlation heatmap
ax2 = fig.add_subplot(gs[1, 0])
im2 = ax2.imshow(
    H_ang.T,
    origin="lower",
    extent=(0, 2*np.pi, 0, 2*np.pi),
    interpolation="nearest",
    aspect="equal",
)
ticks = [0, np.pi/2, np.pi, 3*np.pi/2, 2*np.pi]
ticklabels = ["0", r"$\pi/2$", r"$\pi$", r"$3\pi/2$", r"$2\pi$"]
ax2.set_xticks(ticks, ticklabels)
ax2.set_yticks(ticks, ticklabels)
ax2.set_xlabel(r"$\theta_A$")
ax2.set_ylabel(r"$\theta_B$")
ax2.set_title(r"Joint angular correlation $P(\theta_A,\theta_B)$ (ridge at $\theta_B\approx\theta_A+\pi$)")
fig.colorbar(im2, ax=ax2, fraction=0.046, pad=0.04, label="counts")

# Overlay the ideal opposite-point line θB = θA + π (wrapped)
xx = np.linspace(0, 2*np.pi, 600)
yy = (xx + np.pi) % (2*np.pi)
ax2.plot(xx, yy, linewidth=1.2)

# Panel (1,1): Joint spectrum heatmap
ax3 = fig.add_subplot(gs[1, 1])
im3 = ax3.imshow(
    H_w.T,
    origin="lower",
    extent=(wmin, wmax, wmin, wmax),
    interpolation="nearest",
    aspect="equal",
)
ax3.set_xlabel(r"$\omega_A$")
ax3.set_ylabel(r"$\omega_B$")
ax3.set_title(r"Joint spectrum $P(\omega_A,\omega_B)$ (energy conservation: $\omega_A+\omega_B\approx\omega_p$)")
fig.colorbar(im3, ax=ax3, fraction=0.046, pad=0.04, label="counts")

# Overlay the ideal energy-conservation line ωB = ωp - ωA
wline = np.linspace(wmin, wmax, 600)
ax3.plot(wline, omega_p - wline, linewidth=1.2)

# Add a small annotation block
fig.suptitle(
    "SPDC intuition: singles look 'ring-like' and broadband, but JOINT distributions reveal conservation constraints",
    fontsize=13
)

# Save a copy for download
out_png = "spdc_two_disks_joint_constraints.png"
plt.savefig(out_png, dpi=220, bbox_inches="tight")
plt.show()

out_png


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Re-generate the same synthetic data (seeded) so the panels match prior figure
rng = np.random.default_rng(4)

N_pairs = 12000
r0 = 0.78
sigma_r = 0.035
sigma_theta = 0.05

thetaA = rng.uniform(0, 2*np.pi, size=N_pairs)
thetaB = (thetaA + np.pi + rng.normal(0, sigma_theta, size=N_pairs)) % (2*np.pi)

rA = np.clip(r0 + rng.normal(0, sigma_r, size=N_pairs), 0, 1)
rB = np.clip(r0 + rng.normal(0, sigma_r, size=N_pairs), 0, 1)

xA, yA = rA * np.cos(thetaA), rA * np.sin(thetaA)
xB, yB = rB * np.cos(thetaB), rB * np.sin(thetaB)

omega_p = 2.0
sigma_sum = 0.015
sigma_diff = 0.14

u = rng.normal(0, sigma_sum, size=N_pairs)
v = rng.normal(0, sigma_diff, size=N_pairs)
omegaA = (omega_p/2) + u + v
omegaB = (omega_p/2) + u - v

wmin, wmax = 0.6, 1.4

# Histograms for joint plots
nb_ang = 160
H_ang, *_ = np.histogram2d(thetaA, thetaB, bins=nb_ang, range=[[0, 2*np.pi], [0, 2*np.pi]])

nb_w = 200
H_w, *_ = np.histogram2d(omegaA, omegaB, bins=nb_w, range=[[wmin, wmax], [wmin, wmax]])

# ------------------------------------------------------------
# New 1D "constraint residual" distributions
# Δθ = wrap(thetaB - thetaA - π) into [-π, π]
# δω_sum = ωA + ωB - ωp
# ------------------------------------------------------------
dtheta = thetaB - thetaA - np.pi
dtheta_wrapped = (dtheta + np.pi) % (2*np.pi) - np.pi  # in [-π, π]

domega_sum = (omegaA + omegaB - omega_p)

# Build the new figure with an extra bottom row of small panels
fig = plt.figure(figsize=(12.5, 10.2), constrained_layout=True)
gs = fig.add_gridspec(3, 2, width_ratios=[1, 1.05], height_ratios=[1, 1, 0.55])

# Top row: two disks
ax0 = fig.add_subplot(gs[0, 0])
ax0.scatter(xA, yA, s=1, alpha=0.15)
ax0.set_aspect("equal", adjustable="box")
ax0.set_xlim(-1.02, 1.02)
ax0.set_ylim(-1.02, 1.02)
ax0.set_xticks([]); ax0.set_yticks([])
ax0.set_title("Photon A far-field distribution (ring)")
ax0.add_patch(plt.Circle((0, 0), 1.0, fill=False, linewidth=1.0, alpha=0.5))

ax1 = fig.add_subplot(gs[0, 1])
ax1.scatter(xB, yB, s=1, alpha=0.15)
ax1.set_aspect("equal", adjustable="box")
ax1.set_xlim(-1.02, 1.02)
ax1.set_ylim(-1.02, 1.02)
ax1.set_xticks([]); ax1.set_yticks([])
ax1.set_title("Photon B far-field distribution (ring)")
ax1.add_patch(plt.Circle((0, 0), 1.0, fill=False, linewidth=1.0, alpha=0.5))

# Middle row: joint heatmaps
ax2 = fig.add_subplot(gs[1, 0])
im2 = ax2.imshow(
    H_ang.T,
    origin="lower",
    extent=(0, 2*np.pi, 0, 2*np.pi),
    interpolation="nearest",
    aspect="equal",
)
ticks = [0, np.pi/2, np.pi, 3*np.pi/2, 2*np.pi]
ticklabels = ["0", r"$\pi/2$", r"$\pi$", r"$3\pi/2$", r"$2\pi$"]
ax2.set_xticks(ticks, ticklabels)
ax2.set_yticks(ticks, ticklabels)
ax2.set_xlabel(r"$\theta_A$")
ax2.set_ylabel(r"$\theta_B$")
ax2.set_title(r"Joint angular correlation $P(\theta_A,\theta_B)$ (ridge at $\theta_B\approx\theta_A+\pi$)")
fig.colorbar(im2, ax=ax2, fraction=0.046, pad=0.04, label="counts")
xx = np.linspace(0, 2*np.pi, 600)
yy = (xx + np.pi) % (2*np.pi)
ax2.plot(xx, yy, linewidth=1.2)

ax3 = fig.add_subplot(gs[1, 1])
im3 = ax3.imshow(
    H_w.T,
    origin="lower",
    extent=(wmin, wmax, wmin, wmax),
    interpolation="nearest",
    aspect="equal",
)
ax3.set_xlabel(r"$\omega_A$")
ax3.set_ylabel(r"$\omega_B$")
ax3.set_title(r"Joint spectrum $P(\omega_A,\omega_B)$ (energy conservation: $\omega_A+\omega_B\approx\omega_p$)")
fig.colorbar(im3, ax=ax3, fraction=0.046, pad=0.04, label="counts")
wline = np.linspace(wmin, wmax, 600)
ax3.plot(wline, omega_p - wline, linewidth=1.2)

# Bottom row: residual distributions
ax4 = fig.add_subplot(gs[2, 0])
ax4.hist(dtheta_wrapped, bins=120, density=True)
ax4.grid(True, alpha=0.25)
ax4.set_title(r"Angular residual: $\Delta\theta=\theta_B-\theta_A-\pi$ (wrapped)")
ax4.set_xlabel(r"$\Delta\theta$ (rad)")
ax4.set_ylabel("density")
ax4.axvline(0, linestyle="--", linewidth=1.0)

ax5 = fig.add_subplot(gs[2, 1])
ax5.hist(domega_sum, bins=120, density=True)
ax5.grid(True, alpha=0.25)
ax5.set_title(r"Energy residual: $\omega_A+\omega_B-\omega_p$")
ax5.set_xlabel(r"$\omega_A+\omega_B-\omega_p$")
ax5.set_ylabel("density")
ax5.axvline(0, linestyle="--", linewidth=1.0)

fig.suptitle(
    "SPDC intuition: singles look ring-like/broadband; JOINT distributions reveal conservation + residuals are narrowly peaked",
    fontsize=13
)

out_png = "spdc_two_disks_joint_constraints_with_residuals.png"
plt.savefig(out_png, dpi=220, bbox_inches="tight")
plt.show()

out_png


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# ----------------------------
# Annulus Laplace solver (toy "boundary-coupled global mode")
# u_rr + (1/r) u_r + (1/r^2) u_tt = 0  on r_in < r < r_out
# Dirichlet BC set by Alice (outer boundary) and Bob (inner boundary).
# ----------------------------

def solve_annulus_laplace(a, b, Nr=140, Nth=240, r_in=0.45, r_out=1.0, iters=1200, omega=1.6):
    """
    Solve Laplace's equation on an annulus with boundary conditions:
        u(r_out, θ) = cos(θ - a)
        u(r_in,  θ) = cos(θ - b)
    using SOR on a polar grid.
    """
    theta = np.linspace(0, 2*np.pi, Nth, endpoint=False)
    r = np.linspace(r_in, r_out, Nr)
    dr = r[1] - r[0]
    dth = theta[1] - theta[0]

    # Field
    u = np.zeros((Nr, Nth), dtype=float)

    # Boundary conditions
    u[-1, :] = np.cos(theta - a)   # outer
    u[0,  :] = np.cos(theta - b)   # inner

    # Precompute coefficients for finite differences in polar coordinates
    # Discretization at radial index i (excluding boundaries):
    # u_rr ~ (u_{i+1}-2u_i+u_{i-1})/dr^2
    # (1/r) u_r ~ (1/r_i)(u_{i+1}-u_{i-1})/(2dr)
    # (1/r^2) u_tt ~ (u_{i,j+1}-2u_{i,j}+u_{i,j-1})/(r_i^2 dth^2)
    #
    # Solve for u_i,j in terms of neighbors (Gauss-Seidel form).
    inv_dr2 = 1.0 / (dr*dr)
    inv_2dr = 1.0 / (2.0*dr)
    inv_dth2 = 1.0 / (dth*dth)

    for _ in range(iters):
        # Update interior points
        for i in range(1, Nr-1):
            ri = r[i]
            A = inv_dr2 + (1.0/ri)*inv_2dr      # coeff for u_{i+1,j}
            B = inv_dr2 - (1.0/ri)*inv_2dr      # coeff for u_{i-1,j}
            C = inv_dth2 / (ri*ri)              # coeff for u_{i,j+1} and u_{i,j-1}
            denom = 2.0*inv_dr2 + 2.0*C         # coeff multiplying u_{i,j}

            # Periodic in theta
            jp1 = np.roll(u[i, :], -1)
            jm1 = np.roll(u[i, :],  1)

            # Gauss-Seidel update uses latest u[i-1,:] and u[i+1,:]
            rhs = (A * u[i+1, :] + B * u[i-1, :] + C * (jp1 + jm1))
            u_new = rhs / denom

            # SOR relaxation
            u[i, :] = (1-omega) * u[i, :] + omega * u_new

        # Re-apply boundary conditions (keep exact)
        u[-1, :] = np.cos(theta - a)
        u[0,  :] = np.cos(theta - b)

    return r, theta, u

def annulus_to_cartesian(r, theta, u):
    """
    Convert polar-grid field to Cartesian samples for contourf.
    """
    rr, tt = np.meshgrid(r, theta, indexing="xy")  # rr: (Nth,Nr)
    X = rr * np.cos(tt)
    Y = rr * np.sin(tt)
    # u is (Nr,Nth), transpose to (Nth,Nr) to align with X,Y
    U = u.T
    return X, Y, U

def draw_setting_arrows(ax, r_in, r_out, a, b):
    # Draw boundary circles
    ax.add_patch(plt.Circle((0,0), r_out, fill=False, linewidth=1.2, alpha=0.6))
    ax.add_patch(plt.Circle((0,0), r_in,  fill=False, linewidth=1.2, alpha=0.6))
    # Arrows for a (outer) and b (inner)
    ax.arrow(0, 0, r_out*np.cos(a), r_out*np.sin(a), head_width=0.04, head_length=0.06, length_includes_head=True, lw=1.3)
    ax.arrow(0, 0, r_in*np.cos(b),  r_in*np.sin(b),  head_width=0.04, head_length=0.06, length_includes_head=True, lw=1.3)
    ax.text(1.05*r_out*np.cos(a), 1.05*r_out*np.sin(a), "a (outer)", ha="center", va="center", fontsize=9)
    ax.text(1.12*r_in*np.cos(b),  1.12*r_in*np.sin(b),  "b (inner)", ha="center", va="center", fontsize=9)

# CHSH-like settings for demonstration
a0  = 0.0
ap0 = np.pi/2
b0  = +np.pi/4
bp0 = -np.pi/4

cases = [
    ("a=0°, b=+45°", a0,  b0),
    ("a=0°, b=−45°", a0,  bp0),
    ("a=90°, b=+45°", ap0, b0),
    ("a=90°, b=−45°", ap0, bp0),
]

# Solve and plot
fig = plt.figure(figsize=(12.5, 10.0), constrained_layout=True)
gs = fig.add_gridspec(2, 2, wspace=0.12, hspace=0.12)

r_in, r_out = 0.45, 1.0
levels = 140

last_cf = None
for idx, (title, a, b) in enumerate(cases):
    ax = fig.add_subplot(gs[idx//2, idx%2])
    r, th, u = solve_annulus_laplace(a, b, Nr=140, Nth=240, r_in=r_in, r_out=r_out, iters=900, omega=1.65)
    X, Y, U = annulus_to_cartesian(r, th, u)
    last_cf = ax.contourf(X, Y, U, levels)
    ax.set_aspect("equal", adjustable="box")
    ax.set_xlim(-1.05, 1.05)
    ax.set_ylim(-1.05, 1.05)
    ax.set_xticks([]); ax.set_yticks([])
    ax.set_title(title, fontsize=12)
    draw_setting_arrows(ax, r_in, r_out, a, b)

# Shared colorbar
cbar = fig.colorbar(last_cf, ax=fig.axes, fraction=0.025, pad=0.01)
cbar.set_label("Toy interior field u(r,θ) (harmonic extension of boundaries)", fontsize=11)

fig.suptitle(
    "Annulus model (toy): two boundary conditions (outer=a, inner=b) determine one global interior field",
    fontsize=14
)
out_png = "annulus_boundary_coupling_toy.png"
plt.savefig(out_png, dpi=220, bbox_inches="tight")
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, FFMpegWriter

# ------------------------------------------------------------
# Fast annulus "intensity framing" animation
# Using analytic Laplace solution for the m=2 harmonic of cos^2 boundaries.
# ------------------------------------------------------------

def radial_ABm(r_in, r_out, v_in, v_out, m):
    M = np.array([[r_in**m, r_in**(-m)],
                  [r_out**m, r_out**(-m)]], dtype=float)
    rhs = np.array([v_in, v_out], dtype=float)
    return np.linalg.solve(M, rhs)

def annulus_intensity_U(a, b, r_in=0.45, r_out=1.0, grid=420):
    # boundary cos^2 = 1/2 + 1/2 cos(2(θ-angle)) -> m=2 harmonic amplitude 1/2
    c_out, s_out = 0.5*np.cos(2*a), 0.5*np.sin(2*a)
    c_in,  s_in  = 0.5*np.cos(2*b), 0.5*np.sin(2*b)

    m = 2
    A, B = radial_ABm(r_in, r_out, c_in, c_out, m)  # cos2θ coeff
    C, D = radial_ABm(r_in, r_out, s_in, s_out, m)  # sin2θ coeff

    xs = np.linspace(-r_out, r_out, grid)
    ys = np.linspace(-r_out, r_out, grid)
    X, Y = np.meshgrid(xs, ys, indexing="xy")
    R = np.sqrt(X**2 + Y**2)
    TH = np.arctan2(Y, X)

    mask = (R < r_in) | (R > r_out)

    # radial profiles
    # (avoid 0**(-m) by adding eps; R is masked anyway for small r)
    eps = 1e-12
    cR = A*(R**m) + B/((R+eps)**m)
    sR = C*(R**m) + D/((R+eps)**m)

    U = 0.5 + cR*np.cos(2*TH) + sR*np.sin(2*TH)
    U = np.where(mask, np.nan, U)
    # clamp to [0,1] for "intensity-like" feel
    U = np.clip(U, 0.0, 1.0)
    return U, (-r_out, r_out, -r_out, r_out)

# Spin singlet readout
def P_same(delta):  # delta in radians
    return np.sin(delta/2)**2
def P_opp(delta):
    return np.cos(delta/2)**2
def E_spin(delta):
    return -np.cos(delta)

# Animation parameters
r_in, r_out = 0.45, 1.0
a = 0.0

nframes = 60
deltas = np.linspace(0, np.pi, nframes)  # 0..180°

U0, extent = annulus_intensity_U(a, a, r_in=r_in, r_out=r_out, grid=420)

fig = plt.figure(figsize=(10.6, 5.4), constrained_layout=True)
gs = fig.add_gridspec(1, 2, width_ratios=[1.15, 0.9])

ax_field = fig.add_subplot(gs[0, 0])
ax_plot = fig.add_subplot(gs[0, 1])

im = ax_field.imshow(U0, origin="lower", extent=extent, interpolation="nearest", aspect="equal")
ax_field.set_xlim(-1.05, 1.05)
ax_field.set_ylim(-1.05, 1.05)
ax_field.set_xticks([]); ax_field.set_yticks([])
title = ax_field.set_title("", fontsize=12)

# Add annulus boundary circles once
outer_circle = plt.Circle((0,0), r_out, fill=False, linewidth=1.0, alpha=0.6)
inner_circle = plt.Circle((0,0), r_in,  fill=False, linewidth=1.0, alpha=0.6)
ax_field.add_patch(outer_circle)
ax_field.add_patch(inner_circle)

# Lines as "arrows" (origin to boundary direction)
line_a, = ax_field.plot([0, r_out*np.cos(a)], [0, r_out*np.sin(a)], lw=1.4)
line_b, = ax_field.plot([0, r_in*np.cos(a)],  [0, r_in*np.sin(a)],  lw=1.4)

# Mark bright lobes for cos^2 symmetry (a and a+pi on outer; b and b+pi on inner)
lobes_outer, = ax_field.plot([], [], marker="o", lw=0, markersize=4)
lobes_inner, = ax_field.plot([], [], marker="o", lw=0, markersize=4)

cbar = fig.colorbar(im, ax=ax_field, fraction=0.046, pad=0.04)
cbar.set_label("Intensity-like interior field (toy)", fontsize=10)

# Right panel: probability curves
deg = 180/np.pi
Delta_deg = deltas * deg
ps = P_same(deltas)
po = P_opp(deltas)
e_scaled = (E_spin(deltas)+1)/2  # scaled to [0,1] for overlay

ax_plot.plot(Delta_deg, ps, lw=2, label=r"$P(\mathrm{same})=\sin^2(\Delta/2)$")
ax_plot.plot(Delta_deg, po, lw=2, label=r"$P(\mathrm{opp})=\cos^2(\Delta/2)$")
ax_plot.plot(Delta_deg, e_scaled, lw=1.5, linestyle="--", label=r"(scaled) $E=-\cos\Delta$")
ax_plot.set_xlim(0, 180)
ax_plot.set_ylim(-0.05, 1.05)
ax_plot.grid(True, alpha=0.3)
ax_plot.set_xlabel(r"$\Delta$ (deg)")
ax_plot.set_ylabel("probability")
ax_plot.set_title("Readout (spin singlet)", fontsize=12)
ax_plot.legend(frameon=False, fontsize=9, loc="center right")

m_same = ax_plot.scatter([0], [ps[0]], s=60, zorder=3)
m_opp  = ax_plot.scatter([0], [po[0]], s=60, zorder=3)
readout = ax_plot.text(0.02, 0.05, "", transform=ax_plot.transAxes, fontsize=10,
                       bbox=dict(boxstyle="round", alpha=0.75))

def update(i):
    delta = deltas[i]
    b = a + delta

    U, _ = annulus_intensity_U(a, b, r_in=r_in, r_out=r_out, grid=420)
    im.set_data(U)

    # update lines
    line_a.set_data([0, r_out*np.cos(a)], [0, r_out*np.sin(a)])
    line_b.set_data([0, r_in*np.cos(b)],  [0, r_in*np.sin(b)])

    # update lobe markers
    xo = [r_out*np.cos(a), r_out*np.cos(a+np.pi)]
    yo = [r_out*np.sin(a), r_out*np.sin(a+np.pi)]
    xi = [r_in*np.cos(b),  r_in*np.cos(b+np.pi)]
    yi = [r_in*np.sin(b),  r_in*np.sin(b+np.pi)]
    lobes_outer.set_data(xo, yo)
    lobes_inner.set_data(xi, yi)

    # update title
    title.set_text(f"Annulus interior from cos² boundaries (toy)   a=0°, b={delta*deg:.0f}°   (Δ={delta*deg:.0f}°)")

    # readout markers
    m_same.set_offsets(np.array([[delta*deg, ps[i]]]))
    m_opp.set_offsets(np.array([[delta*deg, po[i]]]))
    readout.set_text(
        f"Δ = {delta*deg:.0f}°\n"
        f"P_same = {ps[i]:.3f}\n"
        f"P_opp  = {po[i]:.3f}\n"
        f"E = {E_spin(delta):+.3f}"
    )

    return []

anim = FuncAnimation(fig, update, frames=nframes, interval=90, blit=False)

out_mp4 = "annulus_intensity_sweep_delta_spin_singlet.mp4"
writer = FFMpegWriter(fps=12, bitrate=1600)
anim.save(out_mp4, writer=writer)

out_mp4
